# GPX1 Discovery Agent
## 03 — RL and Reward Engineering

Goal: wrap the screening campaign as a small RL problem and test whether a learned meta-policy improves over strong hand-designed baselines.

The RL agent does **not** choose directly among thousands of molecules. It chooses whether the next acquisition step should **exploit** predicted activity or **explore** predictive uncertainty.

In [ ]:
from pathlib import Path
import sys
from collections import defaultdict
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..")
sys.path.append(str(PROJECT_ROOT))

from src.featurization import parse_smiles, morgan_fingerprints
from src.campaign import scaffold_split, create_seed_set
from src.environment import GPX1DiscoveryEnv
from src.evaluation import completed_campaign_metrics
from src.models import fit_surrogate

RANDOM_SEED = 42
BUDGET = 40

df = pd.read_csv(PROJECT_ROOT / "data" / "GPX1_curated_for_RL.csv")
mols, valid_mask = parse_smiles(df["PUBCHEM_EXT_DATASOURCE_SMILES"])
clean = df.loc[valid_mask].copy().reset_index(drop=True)

X = morgan_fingerprints(mols)
y = clean["label"].to_numpy(dtype=int)
scaffolds = clean["scaffold"].astype(str).to_numpy()

campaign_idx, validation_idx = scaffold_split(
    X, y, scaffolds,
    test_size=0.20,
    random_state=RANDOM_SEED,
)

X_campaign = X[campaign_idx]
y_campaign = y[campaign_idx]
campaign_scaffolds = scaffolds[campaign_idx]

X_validation = X[validation_idx]
y_validation = y[validation_idx]

### Scientific reward

Reward combines:
- activity discovery,
- novel active scaffold discovery,
- information gain from reduced pool entropy.

Raw information gain is much smaller numerically than the hit terms, so the environment uses:

`scaled_information_gain = tanh(100 × information_gain)`

This keeps the information term bounded and visible to the learner.

### Observation and actions

**Observation**
1. campaign progress
2. cumulative hit rate
3. recent hit rate
4. mean pool entropy
5. active scaffold diversity

**Actions**
- `0` = exploit highest predicted activity
- `1` = explore highest predictive entropy

In [ ]:
STATE_BINS = [
    np.array([0.33, 0.67]),
    np.array([0.25, 0.60]),
    np.array([0.25, 0.60]),
    np.array([0.33, 0.66]),
    np.array([0.60, 0.85]),
]

def discretize_state(state):
    return tuple(
        np.digitize(value, bins)
        for value, bins in zip(state, STATE_BINS)
    )

### Q-learning

In [ ]:
N_ACTIONS = 2

def choose_q_action(Q, state_key, training_epsilon, rng):
    if rng.random() < training_epsilon:
        return int(rng.integers(N_ACTIONS))

    q_values = Q[state_key]
    best_actions = np.flatnonzero(
        q_values == q_values.max()
    )
    return int(rng.choice(best_actions))


def train_q_agent(
    reward_config,
    n_episodes=40,
    starting_seed=20,
    alpha=0.20,
    gamma=0.95,
    epsilon_start=1.0,
    epsilon_end=0.05,
):
    Q = defaultdict(
        lambda: np.zeros(N_ACTIONS, dtype=float)
    )
    rows = []
    rng = np.random.default_rng(RANDOM_SEED)

    for episode in range(n_episodes):
        campaign_seed = starting_seed + episode
        seed_idx, pool_idx = create_seed_set(
            y_campaign,
            campaign_seed,
        )

        env = GPX1DiscoveryEnv(
            X_campaign,
            y_campaign,
            campaign_scaffolds,
            budget=BUDGET,
            **reward_config,
        )

        state = env.reset(seed_idx, pool_idx)
        state_key = discretize_state(state)

        fraction = episode / max(
            n_episodes - 1,
            1,
        )
        training_epsilon = (
            epsilon_start
            + fraction * (
                epsilon_end - epsilon_start
            )
        )

        total_reward = 0.0

        while True:
            action = choose_q_action(
                Q,
                state_key,
                training_epsilon,
                rng,
            )

            next_state, reward, done, _ = env.step(
                action
            )
            next_key = discretize_state(
                next_state
            )

            target = (
                reward
                if done
                else reward
                + gamma * np.max(Q[next_key])
            )

            Q[state_key][action] += alpha * (
                target - Q[state_key][action]
            )

            total_reward += reward
            state_key = next_key

            if done:
                break

        history = pd.DataFrame(env.history)

        rows.append({
            "episode": episode + 1,
            "campaign_seed": campaign_seed,
            "training_epsilon": training_epsilon,
            "total_reward": total_reward,
            "hits": int(
                history["observed_label"].sum()
            ),
            "novel_scaffolds": int(
                history[
                    "novel_active_scaffold"
                ].sum()
            ),
            "explore_actions": int(
                (history["action"] == 1).sum()
            ),
        })

    return Q, pd.DataFrame(rows)

### Train the initial agent

In [ ]:
INITIAL_REWARD = {
    "activity_reward": 2.0,
    "novelty_reward": 1.0,
    "information_reward": 0.5,
}

Q_initial, training_history = train_q_agent(
    INITIAL_REWARD,
    n_episodes=40,
    starting_seed=20,
)

print("States learned:", len(Q_initial))
print(
    "Mean reward first 10:",
    training_history["total_reward"].head(10).mean(),
)
print(
    "Mean reward last 10:",
    training_history["total_reward"].tail(10).mean(),
)
print(
    "Mean hits first 10:",
    training_history["hits"].head(10).mean(),
)
print(
    "Mean hits last 10:",
    training_history["hits"].tail(10).mean(),
)

### Frozen-policy evaluation

The Q-table is frozen during evaluation. If the policy encounters a discretized state that was never observed during training, it falls back conservatively to exploitation.

In [ ]:
def evaluate_q_agent(
    Q_agent,
    reward_config,
    campaign_seed,
):
    seed_idx, pool_idx = create_seed_set(
        y_campaign,
        campaign_seed,
    )

    env = GPX1DiscoveryEnv(
        X_campaign,
        y_campaign,
        campaign_scaffolds,
        budget=BUDGET,
        **reward_config,
    )

    state = env.reset(seed_idx, pool_idx)
    total_reward = 0.0
    covered_steps = 0

    while True:
        state_key = discretize_state(state)

        if state_key in Q_agent:
            covered_steps += 1
            action = int(
                np.argmax(Q_agent[state_key])
            )
        else:
            action = 0

        state, reward, done, _ = env.step(action)
        total_reward += reward

        if done:
            break

    history = pd.DataFrame(env.history)

    metrics = completed_campaign_metrics(
        history,
        seed_idx,
        X_campaign,
        y_campaign,
        campaign_scaffolds,
        X_validation,
        y_validation,
        fit_surrogate,
    )

    metrics["total_reward"] = total_reward
    metrics["explore_actions"] = int(
        (history["action"] == 1).sum()
    )
    metrics["q_state_coverage"] = (
        covered_steps / len(history)
    )
    return metrics

### Reward ablation

In [ ]:
REWARD_CONFIGS = {
    "Hit-Centric": {
        "activity_reward": 2.0,
        "novelty_reward": 0.0,
        "information_reward": 0.0,
    },
    "Diversity-Aware": {
        "activity_reward": 2.0,
        "novelty_reward": 2.0,
        "information_reward": 0.0,
    },
    "Learning-Balanced": {
        "activity_reward": 2.0,
        "novelty_reward": 1.0,
        "information_reward": 1.0,
    },
}

reward_agents = {}

for name, config in REWARD_CONFIGS.items():
    print("Training:", name)

    Q_agent, _ = train_q_agent(
        config,
        n_episodes=40,
        starting_seed=80,
    )
    reward_agents[name] = Q_agent

In [ ]:
rows = []

for name, Q_agent in reward_agents.items():
    for campaign_seed in range(120, 140):
        metrics = evaluate_q_agent(
            Q_agent,
            REWARD_CONFIGS[name],
            campaign_seed,
        )
        rows.append({
            "reward_objective": name,
            "campaign": campaign_seed,
            **metrics,
        })

reward_ablation = pd.DataFrame(rows)

(
    reward_ablation
    .groupby("reward_objective")
    .agg(
        mean_hits=("hits", "mean"),
        hit_sd=("hits", "std"),
        mean_active_scaffolds=(
            "active_scaffolds",
            "mean",
        ),
        mean_validation_pr_auc=(
            "validation_pr_auc",
            "mean",
        ),
        mean_explore_actions=(
            "explore_actions",
            "mean",
        ),
    )
    .reset_index()
)

## Interpretation

The strongest result is **not** that RL beats exploitation.

Instead:
- hit-centric reward produces an almost purely exploitative policy;
- adding information value increases exploration;
- uncertainty exploration does not automatically produce chemical scaffold novelty;
- reward design and action-space design are separate scientific modeling problems.

A stronger next version would add an explicit chemical-diversity action and replace score-based uncertainty with ensemble disagreement.